In [3]:
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pprint import pprint
from pathlib import Path
from tqdm.auto import tqdm

# Fill this from the DANDI URL.
# Example DANDI URL usually looks like:
# https://dandiarchive.org/dandiset/000XXX/draft
DANDISET_ID = "000006"
VERSION = "draft"

MAX_ASSETS_TO_SCAN = 20
ASSET_INDEX_TO_OPEN = 0

client = DandiAPIClient.for_dandi_instance("dandi")
dandiset = client.get_dandiset(DANDISET_ID, VERSION)

print("Dandiset:", dandiset.identifier)
print("Version:", VERSION)

# Dandiset metadata
try:
    dandiset_meta = dandiset.get_raw_metadata()
except Exception:
    dandiset_meta = dandiset.get_metadata()

print("Dandiset metadata type:", type(dandiset_meta))
pprint(dandiset_meta if isinstance(dandiset_meta, dict) else str(dandiset_meta)[:2000])


Dandiset: 000006
Version: draft
Dandiset metadata type: <class 'dict'>
{'@context': 'https://raw.githubusercontent.com/dandi/schema/master/releases/0.6.0/context.json',
 'about': [{'identifier': 'http://purl.obolibrary.org/obo/UBERON_0016634',
            'name': 'premotor cortex',
            'schemaKey': 'Anatomy'},
           {'identifier': 'https://purl.brain-bican.org/ontology/dhbao/DHBA_12773',
            'name': 'pyramidal tract',
            'schemaKey': 'Anatomy'}],
 'access': [{'contactPoint': {'schemaKey': 'ContactPoint'},
             'schemaKey': 'AccessRequirements',
             'status': 'dandi:OpenAccess'}],
 'assetsSummary': {'approach': [{'name': 'electrophysiological approach',
                                 'schemaKey': 'ApproachType'},
                                {'name': 'behavioral approach',
                                 'schemaKey': 'ApproachType'}],
                   'dataStandard': [{'identifier': 'RRID:SCR_015242',
                               

In [5]:
dat=pd.read_csv('/home/maria/mousehash/src/DANDI-agent/mousehash_dandi_scan/mousehash_role_matrix.csv')

In [6]:
feature_sums = dat.drop(columns=["dandiset_id", "name"], errors="ignore").sum(axis=0).sort_values(ascending=False)

In [7]:
feature_sums

metadata.number_of_bytes                  819
metadata.number_of_files                  819
metadata.license                          768
metadata.number_of_subjects               537
metadata.data_standard                    517
metadata.species                          497
neural_data.calcium                       455
metadata.probe/electrode/imaging_plane    358
neural_data.spikes                        298
metadata.brain_area                       292
stimuli.interventions.surgical            274
time_organization.continuous_time         260
metadata.preprocessing_info               255
behavior.behavioral_events                177
stimuli.sensory.visual                    149
neural_data.images                        137
conditions.task_labels                    135
time_organization.events                  112
behavior.kinematics                       102
time_organization.frames                  100
neural_data.lfp                            81
neural_data.intracellular_recordin

In [8]:
import json
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd


SCAN_PATH = Path(
    "/home/maria/mousehash/src/DANDI-agent/"
    "mousehash_dandi_scan/dandi_mousehash_scan_full.json"
)

OUT_DIR = Path(
    "/home/maria/mousehash/src/DANDI-agent/mousehash_dandi_scan"
)

records = json.loads(SCAN_PATH.read_text(encoding="utf-8"))


def walk_assets_summary_schema_keys(obj, path="assetsSummary"):
    """
    Recursively walk assetsSummary.

    Whenever a dict has schemaKey, record:
      schemaKey
      path
      all fields on that dict
    """
    rows = []

    if isinstance(obj, dict):
        schema_key = obj.get("schemaKey")

        if schema_key is not None:
            row = {
                "schemaKey": schema_key,
                "path": path,
            }

            for k, v in obj.items():
                if isinstance(v, (str, int, float, bool)) or v is None:
                    row[k] = v
                else:
                    row[k] = json.dumps(v, ensure_ascii=False, default=str)

            rows.append(row)

        for k, v in obj.items():
            rows.extend(
                walk_assets_summary_schema_keys(
                    v,
                    path=f"{path}.{k}",
                )
            )

    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            rows.extend(
                walk_assets_summary_schema_keys(
                    item,
                    path=f"{path}[{i}]",
                )
            )

    return rows


schema_rows = []

for rec in records:
    raw_meta = rec.get("raw_metadata", {})
    summary = raw_meta.get("assetsSummary", {}) or {}

    rows = walk_assets_summary_schema_keys(summary)

    for row in rows:
        row["dandiset_id"] = rec.get("dandiset_id")
        row["dandiset_name"] = rec.get("name")

    schema_rows.extend(rows)

schema_df = pd.DataFrame(schema_rows)

schema_df.head()

,schemaKey,path,species,approach,dataStandard,numberOfBytes,numberOfFiles,numberOfSubjects,variableMeasured,measurementTechnique,dandiset_id,dandiset_name,name,identifier,numberOfCells,numberOfSamples
0,AssetsSummary,assetsSummary,"[{""name"": ""House mouse"", ""schemaKey"": ""Species...","[{""name"": ""electrophysiological approach"", ""sc...","[{""name"": ""Neurodata Without Borders (NWB)"", ""...",2.559248e+12,101.0,16.0,"[""DecompositionSeries"", ""LFP"", ""Units"", ""Posit...","[{""name"": ""signal filtering technique"", ""schem...",DANDI:000003,Physiological Properties and Behavioral Correl...,NaN,NaN,NaN,NaN
1,SpeciesType,assetsSummary.species[0],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DANDI:000003,Physiological Properties and Behavioral Correl...,House mouse,http://purl.obolibrary.org/obo/NCBITaxon_10090,NaN,NaN
2,ApproachType,assetsSummary.approach[0],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DANDI:000003,Physiological Properties and Behavioral Correl...,electrophysiological approach,NaN,NaN,NaN
3,ApproachType,assetsSummary.approach[1],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DANDI:000003,Physiological Properties and Behavioral Correl...,behavioral approach,NaN,NaN,NaN
4,StandardsType,assetsSummary.dataStandard[0],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DANDI:000003,Physiological Properties and Behavioral Correl...,Neurodata Without Borders (NWB),RRID:SCR_015242,NaN,NaN
